In [0]:
%sql
CREATE TABLE IF NOT EXISTS fk_ce_2026_07_1382.bronze.stores;

COPY INTO fk_ce_2026_07_1382.bronze.stores
FROM (
    SELECT
        *,
        _metadata.file_path AS file_path,
        current_timestamp() AS ingestion_timestamp
    FROM '/Volumes/fk_ce_2026_07_1382/landing/raw/stores/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS (
    'header' = 'true'
)
COPY_OPTIONS (
    'mergeSchema' = 'true'
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS fk_ce_2026_07_1382.bronze.customers;

COPY INTO fk_ce_2026_07_1382.bronze.customers
FROM (
    SELECT
        *,
        _metadata.file_path AS file_path,
        current_timestamp() AS ingestion_timestamp
    FROM '/Volumes/fk_ce_2026_07_1382/landing/raw/customers/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS (
    'header' = 'true'
)
COPY_OPTIONS (
    'mergeSchema' = 'true'
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS fk_ce_2026_07_1382.bronze.products;

COPY INTO fk_ce_2026_07_1382.bronze.products
FROM (
    SELECT
        *,
        _metadata.file_path AS file_path,
        current_timestamp() AS ingestion_timestamp
    FROM '/Volumes/fk_ce_2026_07_1382/landing/raw/products/'
)
FILEFORMAT = JSON
FORMAT_OPTIONS (
    'header' = 'true',
    'multiline' = 'true'
)
COPY_OPTIONS (
    'mergeSchema' = 'true'
);

In [0]:
from pyspark.sql.functions import *

df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("header", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", "/Volumes/fk_ce_2026_07_1382/ops/pipeline_state/orders_schema/")
    .load("/Volumes/fk_ce_2026_07_1382/landing/raw/orders/")
    .select(
        col("*"),
        col("_metadata.file_path").alias("file_path"),
        current_timestamp().alias("ingestion_timestamp")
    )
)

(
    df.writeStream.format("delta")
    .option("checkpointLocation", "/Volumes/fk_ce_2026_07_1382/ops/pipeline_state/orders_checkpoint/")
    .trigger(availableNow=True)
    .toTable("fk_ce_2026_07_1382.bronze.orders")
)